# 01 Data Loading
Load raw Olist datasets from the data/raw folder.

In [4]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipelines.data_pipeline import run_pipeline

PROCESSED_DIR = ROOT / "data" / "processed"
REPORT_PATH = PROCESSED_DIR / "data_quality_report.json"

In [5]:
run_result = run_pipeline()
quality_report = run_result["quality_report"]

print("Critical failures:", len(quality_report["critical_failures"]))
for msg in quality_report["critical_failures"]:
    print("-", msg)

if quality_report["critical_failures"]:
    raise RuntimeError("Pre-EDA quality gate failed. Resolve failures before EDA.")

print("Quality gate passed.")

Critical failures: 0
Quality gate passed.


In [6]:
with open(REPORT_PATH, "r", encoding="utf-8") as f:
    report = json.load(f)

display(pd.DataFrame(report["tables"]).T[["rows_raw", "rows_clean", "duplicates_removed"]])

clean_files = sorted(PROCESSED_DIR.glob("*_clean.parquet"))
print("Cleaned tables:")
for file_path in clean_files:
    df = pd.read_parquet(file_path)
    print(f"- {file_path.name}: {df.shape}")

,rows_raw,rows_clean,duplicates_removed
customers,99441,99441,0
orders,99441,99441,0
order_items,112650,112650,0
order_payments,103886,103886,0
order_reviews,99224,98410,814
products,32951,32951,0
sellers,3095,3095,0
geolocation,1000163,1000163,0


Cleaned tables:
- olist_customers_clean.parquet: (99441, 5)
- olist_geolocation_clean.parquet: (1000163, 6)
- olist_order_items_clean.parquet: (112650, 11)
- olist_order_payments_clean.parquet: (103886, 7)
- olist_order_reviews_clean.parquet: (98410, 8)
- olist_orders_clean.parquet: (99441, 9)
- olist_products_clean.parquet: (32951, 11)
- olist_sellers_clean.parquet: (3095, 4)
